In [7]:
!pip install newspaper3k nltk networkx scikit-learn transformers torch

In [8]:
!pip install lxml_html_clean
import nltk
nltk.download('punkt')

from newspaper import Article
from nltk.tokenize import sent_tokenize
import numpy as np

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
url = "https://www.usatoday.com/story/news/politics/2025/06/13/pete-hegseth-pentagon-invade-greenland-plan/84188458007/"

article = Article(url)
article.download()
article.parse()

text = article.text

print("Article length:", len(text))
print("\nPreview:\n")
print(text[:800])

Article length: 1684

Preview:

June 13, 2025, 2:33 p.m. ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct? Because I sure as hell hope that it is not your testimony," Turner dug in.

"We look forward to working with Greenland to ensure that it is secured from any potential threats," Hegseth said.

President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he


In [10]:
import nltk
nltk.download('punkt_tab')

sentences = sent_tokenize(text)

print("Total sentences:", len(sentences))

[nltk_data] Downloading package punkt_tab to /root/nltk_data...


Total sentences: 14


[nltk_data]   Package punkt_tab is already up-to-date!


1.Extractive Summary — TextRank

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sentences)

similarity_matrix = (tfidf_matrix * tfidf_matrix.T).toarray()

graph = nx.from_numpy_array(similarity_matrix)

scores = nx.pagerank(graph)

ranked_sentences = sorted(((scores[i], s) for i, s in enumerate(sentences)), reverse=True)

summary_len = max(3, int(len(sentences) * 0.2))

textrank_summary = [ranked_sentences[i][1] for i in range(summary_len)]

print("TEXT RANK SUMMARY\n")

for s in textrank_summary:
    print(s)

TEXT RANK SUMMARY

"It is not your testimony today that there are plans at the Pentagon for taking by force or invading Greenland, correct?
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.


2.Frequency-Based Sentence Scoring

In [12]:
from nltk.corpus import stopwords
from collections import defaultdict
import string

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

words = nltk.word_tokenize(text.lower())

freq = defaultdict(int)

for word in words:
    if word not in stop_words and word not in string.punctuation:
        freq[word] += 1

sentence_scores = {}

for sent in sentences:
    for word in nltk.word_tokenize(sent.lower()):
        if word in freq:
            if sent not in sentence_scores:
                sentence_scores[sent] = freq[word]
            else:
                sentence_scores[sent] += freq[word]

ranked = sorted(sentence_scores, key=sentence_scores.get, reverse=True)

freq_summary = ranked[:summary_len]

print("FREQUENCY SUMMARY\n")

for s in freq_summary:
    print(s)

FREQUENCY SUMMARY

Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."
During a March visit to Pituffik Space Base, the U.S. base on Greenland, Vice President JD Vance accused Denmark of "failing" to protect the Arctic island while downplaying Trump's threats to take it over by force.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


3.Abstractive Summary (Transformer: BART)

In [14]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

inputs = tokenizer([text[:3000]], max_length=1024, return_tensors='pt', truncation=True)

# Generate Summary
summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=200, min_length=80, early_stopping=True)
summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("ABSTRACTIVE SUMMARY (BART)\n")
print(summary_text)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

ABSTRACTIVE SUMMARY (BART)

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island. President Donald Trump has declined to rule out force in his pledge to "get Greenland," although he has said it won't be necessary. The Pentagon reportedly plans to move its oversight of the Arctic island from U.S. European Command to U.s. Northern Command.


4.Lead-3 Summary

In [15]:
lead3 = sentences[:3]

print("LEAD-3 SUMMARY\n")

for s in lead3:
    print(s)

LEAD-3 SUMMARY

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.
Asked by Republican Rep. Mike Turner at a June 12 House Armed Services Committee hearing to confirm whether there are plans to invade Greenland, Hegseth said, "The Pentagon has plans for any number of contingencies."


5.Manual Compression (about 20% Sentences)

In [16]:
compression_len = int(len(sentences) * 0.2)

manual_summary = sentences[:compression_len]

print("MANUAL COMPRESSION SUMMARY\n")

for s in manual_summary:
    print(s)

MANUAL COMPRESSION SUMMARY

June 13, 2025, 2:33 p.m.
ET

Defense Secretary Pete Hegseth said the Pentagon has plans for multiple "contingencies" in Greenland – including an invasion of the island.


6.LLM Summary (Using HuggingFace Large Model)

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/pegasus-xsum")
model = AutoModelForSeq2SeqLM.from_pretrained("google/pegasus-xsum")

inputs = tokenizer([text[:3000]], max_length=1024, return_tensors='pt', truncation=True)

# Generate Summary
summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=120, min_length=40, early_stopping=True)
llm_summary_text = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("LLM SUMMARY\n")
print(llm_summary_text)

tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-xsum
Key                                  | Status  | 
-------------------------------------+---------+-
model.encoder.embed_positions.weight | MISSING | 
model.decoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/259 [00:00<?, ?B/s]

LLM SUMMARY

The Pentagon says it has plans for an invasion of Greenland, but it won't say whether it's planning to take it over by force or not, according to a report by USA TODAY.
